<a href="https://colab.research.google.com/github/wachirapatsiri/personal-finance-pipeline-dashboard/blob/main/Dashboard_income_expense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dashboard รายรับ-รายจ่าย (Frontend ผ่าน Google Colab)

รันเซลล์ตามลำดับ 1 → 4 แล้วจะได้ลิงก์เว็บ (`*.gradio.live`)
หน้าเว็บมีแค่ช่องใส่ไฟล์ CSV กับปุ่ม Submit กดแล้วระบบจะรัน Pipeline ทำความสะอาดข้อมูล
จัดหมวดหมู่ด้วย Fuzzy Matching แล้วแสดง Dashboard ให้ในหน้าเดียวกัน

ไฟล์ CSV ต้องมีคอลัมน์: `transaction_id`, `date`, `description`, `amount`


In [ ]:
# ===== 1) ติดตั้ง Library + ฟอนต์ไทย =====
!pip install -q gradio thefuzz
!wget -q -O thsarabunnew-webfont.ttf https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.2 MB/s eta 0:00:00


In [ ]:
# ===== 2) Import + ตั้งค่าฟอนต์ไทยให้ Matplotlib =====
import shutil
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import gradio as gr
from thefuzz import process, fuzz

mpl.use('Agg')   # สำคัญ: วาดกราฟส่งให้เว็บ ไม่ต้องเปิดหน้าต่างแสดงผล
mpl.font_manager.fontManager.addfont('thsarabunnew-webfont.ttf')
mpl.rc('font', family='TH Sarabun New', size=14)
mpl.rc('axes', unicode_minus=False)


In [ ]:
# ===== 3) Pipeline: Dictionary หมวดหมู่ + ฟังก์ชันตรวจสอบ (Validation) + ทำความสะอาดข้อมูล =====
merchant_categories = {
    # ==== ร้านสะดวกซื้อ / ซูเปอร์มาร์เก็ต & ตลาด ====
    '7-eleven': 'Supermarket',
    'seven eleven': 'Supermarket',
    'lotus': 'Supermarket',
    "lotus's": 'Supermarket',
    'ตลาดนัดหน้าหอ': 'Supermarket',

    # ==== การเดินทาง (Transportation) ====
    'grab': 'Transportation',
    'วินมอเตอร์ไซค์': 'Transportation',
    'รถเมล์ ขสมก.': 'Transportation',

    # ==== ช้อปปิ้งออนไลน์ & เสื้อผ้า ====
    'shopee': 'Online Shopping',
    'shopee thailand': 'Online Shopping',
    'lazada': 'Online Shopping',
    'uniqlo': 'Shopping & Fashion',

    # ==== สาธารณูปโภค & ที่พักอาศัย ====
    'true': 'Utilities',
    'true corporation': 'Utilities',
    'การไฟฟ้าส่วนภูมิภาค': 'Utilities',
    'การประปาส่วนภูมิภาค': 'Utilities',
    'ค่าหอพัก แสงจันทร์อพาร์ตเมนต์': 'Housing',

    # ==== สุขภาพและความงาม ====
    'boots': 'Health & Beauty',

    # ==== อาหารและเครื่องดื่ม ====
    'starbucks': 'Food & Beverage',
    'amazon café': 'Food & Beverage',
    'amazon cafe': 'Food & Beverage',
    'ร้านลุงมี ก๋วยเตี๋ยวเรือ': 'Food & Beverage',
    'ร้านก๋วยเตี๋ยวป้าแดง': 'Food & Beverage',
    'ร้านส้มตำหน้าม.': 'Food & Beverage',
    'ร้านข้าวแกงป้านิด': 'Food & Beverage',
    'ร้านกาแฟหน้าปากซอย': 'Food & Beverage',
    'ฟู้ดคอร์ทมหาลัย': 'Food & Beverage',

    # ==== บันเทิง ====
    'netflix': 'Entertainment',
    'netflix.com': 'Entertainment',

    # ==== รายรับ / เงินโอนเข้า ====
    'รายได้ฟรีแลนซ์': 'Income',
    'ดอกเบี้ยเงินฝาก': 'Income',
    'เงินคืนภาษี': 'Income',
    'ค่าคอมมิชชั่น': 'Income',
    'cashback shopee': 'Income',   # เงินคืนจาก Shopee นับเป็นรายรับ (ไม่ใช่การซื้อของ)
    'เงินเดือน': 'Income',
    'เงินเดือน/ค่าขนมรายเดือน': 'Income',
    'โอนเงินจาก': 'Income',   # โอนเงินจาก แม่ / เพื่อน / ใครก็ตาม -> รวมเป็น "เงินโอน" อยู่ในหมวดรายรับ

    # ==== เงินโอนออก (รวมทุกคนเป็น "เงินโอน" และนับเป็นรายจ่ายเท่านั้น) ====
    'โอนเงินให้': 'Transfer Out',
}
standard_merchants = list(merchant_categories.keys())

# ชื่อที่ใช้แสดงผลแทนชื่อร้านมาตรฐาน (รวมเงินโอนเข้า/ออกทุกคนให้เหลือชื่อเดียวคือ "เงินโอน")
display_names = {
    'โอนเงินจาก': 'เงินโอน', 'โอนเงินให้': 'เงินโอน',
    # รวมชื่อที่สะกดต่างกันของร้านเดียวกัน ให้แสดงเป็นชื่อเดียวในกราฟ
    '7-eleven': '7-Eleven', 'seven eleven': '7-Eleven',
    'lotus': "Lotus's", "lotus's": "Lotus's",
    'grab': 'Grab',
    'shopee': 'Shopee', 'shopee thailand': 'Shopee',
    'true': 'True', 'true corporation': 'True',
    'starbucks': 'Starbucks',
    'amazon café': 'Amazon Café', 'amazon cafe': 'Amazon Café',
    'netflix': 'Netflix', 'netflix.com': 'Netflix',
    'boots': 'Boots',
    'cashback shopee': 'Cashback Shopee',
    'เงินเดือน/ค่าขนมรายเดือน': 'เงินเดือน',   # แสดงเป็น "เงินเดือน" อย่างเดียว
}

# ---- นิยามประเภทกระแสเงิน (ใช้ร่วมกันทั้ง KPI, กราฟ, และตาราง เพื่อให้ตัวเลขตรงกันทุกที่) ----
#   income       = หมวด Income (รวมเงินโอนเข้าทั้งหมด) -> นับเป็นรายรับ
#   expense      = ทุกหมวดที่เหลือ (รวม Others และ Transfer Out) -> นับเป็นรายจ่าย
def classify_flow(category):
    if category == 'Income':
        return 'income'
    return 'expense'


# ---- นโยบายจำนวนเงินติดลบ (ระบบเดาไม่ได้ว่าเป็นรายการคืนเงินหรือคีย์ผิด) ----
#   'exclude'     = ไม่นำมาคำนวณ แต่เก็บไว้ในตาราง flagged ให้ตรวจสอบ (ค่าเริ่มต้น ปลอดภัยที่สุด)
#   'abs'         = แปลงเป็นค่าบวก (เหมาะเมื่อมั่นใจว่าเป็นการคีย์เครื่องหมายผิด)
NEGATIVE_POLICY = 'exclude'
NEGATIVE_NOTE = {
    'exclude': '(ไม่นำมาคำนวณ แนะนำให้ตรวจสอบต้นฉบับ)',
    'abs': '(แปลงเป็นค่าบวก แนะนำให้ตรวจสอบต้นฉบับ)',
}
REQUIRED_COLS = ['transaction_id', 'date', 'description', 'amount']


def read_csv_safe(path):
    """อ่าน CSV เผื่อไฟล์เป็น utf-8-sig หรือ cp874 (ไฟล์ไทยที่เซฟจาก Excel)"""
    for enc in ('utf-8-sig', 'utf-8', 'cp874'):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError('อ่านไฟล์ไม่ได้ ลองเซฟเป็น CSV UTF-8 แล้วอัปโหลดใหม่')


def parse_date_robust(s):
    candidate_formats = ['%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y']
    result = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    for fmt in candidate_formats:
        remaining = result.isna()
        if not remaining.any():
            break
        result.loc[remaining] = pd.to_datetime(s.loc[remaining], format=fmt, errors='coerce')
    return result


def match_and_categorize(merchant_name):
    # 1) ตรงเป๊ะกับ dictionary -> ใช้เลย (กัน 'cashback shopee' ถูกดึงไปเป็น 'shopee')
    if merchant_name in merchant_categories:
        best_match = merchant_name
    else:
        # 2) Fuzzy: ถ้าคะแนนเท่ากัน ให้เลือกคีย์ที่ยาวกว่า (เฉพาะเจาะจงกว่า)
        scored = process.extract(merchant_name, standard_merchants,
                                 scorer=fuzz.token_set_ratio, limit=None)
        best_match, score = max(scored, key=lambda x: (x[1], len(x[0])))
        if score < 70:
            return pd.Series([merchant_name, 'Others'])
    return pd.Series([display_names.get(best_match, best_match), merchant_categories[best_match]])


def validate_and_clean(raw_df):
    """
    ตรวจสอบความถูกต้องของโครงสร้างข้อมูล ก่อนทำความสะอาดและจัดหมวดหมู่
    เช็ก: ค่าว่างในคอลัมน์จำเป็น, จำนวนเงินไม่ใช่ตัวเลข, จำนวนเงินติดลบ,
          รูปแบบวันที่ผิด, transaction_id ซ้ำ

    คืนค่า: (df_clean, report:dict, flagged_df) ทุกแถวที่ถูกตัดหรือถูกแก้จะ
    ถูกเก็บไว้ใน flagged_df พร้อมคอลัมน์ validation_issue อธิบายเหตุผล
    """
    df = raw_df.copy()
    total_raw = len(df)
    flagged_frames = []

    def flag(mask, reason):
        rows = df.loc[mask].copy()
        if len(rows):
            rows['validation_issue'] = reason
            flagged_frames.append(rows)
        return rows

    # 1) ค่าว่างในคอลัมน์ที่จำเป็น
    null_mask = df[REQUIRED_COLS].isna().any(axis=1)
    null_rows = flag(null_mask, 'มีค่าว่างในคอลัมน์ที่จำเป็น (transaction_id/date/description/amount)')
    df = df[~null_mask]

    # Text Normalization
    df['description_cleaned'] = df['description'].astype(str).str.strip().str.lower()

    # 2) จำนวนเงิน: แปลงเป็นตัวเลข แล้วเช็กว่าแปลงไม่ได้หรือไม่
    df['amount'] = pd.to_numeric(
        df['amount'].astype(str).str.replace(r'[^\d.\-]', '', regex=True),
        errors='coerce'
    )
    invalid_amount_mask = df['amount'].isna()
    flag(invalid_amount_mask, 'จำนวนเงินไม่ใช่ตัวเลข')
    df = df[~invalid_amount_mask]

    # 3) จำนวนเงินติดลบ: จัดการตาม NEGATIVE_POLICY และบันทึกไว้ให้ตรวจสอบทุกกรณี
    negative_mask = df['amount'] < 0
    negative_count = int(negative_mask.sum())
    if NEGATIVE_POLICY not in NEGATIVE_NOTE:
        raise ValueError(f"NEGATIVE_POLICY ต้องเป็นหนึ่งใน {list(NEGATIVE_NOTE)}")
    flag(negative_mask, f"จำนวนเงินติดลบ {NEGATIVE_NOTE[NEGATIVE_POLICY]}")
    if NEGATIVE_POLICY == 'exclude':
        df = df[~negative_mask]
    else:  # 'abs'
        df.loc[negative_mask, 'amount'] = df.loc[negative_mask, 'amount'].abs()

    # 4) รูปแบบวันที่
    date_str = df['date'].astype(str).str.strip()
    df['date_cleaned'] = parse_date_robust(date_str)
    invalid_date_mask = df['date_cleaned'].isna()
    # แถวที่วันที่ไม่ได้อยู่ในรูปแบบมาตรฐาน YYYY-MM-DD แต่แปลงได้ -> นับเป็น "ถูกแปลง format"
    is_iso = pd.to_datetime(date_str, format='%Y-%m-%d', errors='coerce').notna()
    date_reformatted = int((~is_iso & ~invalid_date_mask).sum())
    flag(invalid_date_mask, 'รูปแบบวันที่ไม่ถูกต้อง (ไม่ตรง YYYY-MM-DD, DD/MM/YYYY, DD-MM-YYYY)')
    df = df[~invalid_date_mask]

    # 5) transaction_id ซ้ำ (เก็บรายการแรกไว้)
    dup_mask = df.duplicated(subset=['transaction_id'], keep='first')
    flag(dup_mask, 'transaction_id ซ้ำ (เก็บรายการแรก ตัดรายการซ้ำออก)')
    df = df[~dup_mask]

    # จัดหมวดหมู่ด้วย Fuzzy Matching
    if len(df):
        df[['description_standard', 'category']] = df['description_cleaned'].apply(match_and_categorize)
        df['flow'] = df['category'].map(classify_flow)
    else:
        for col in ('description_standard', 'category', 'flow'):
            df[col] = pd.Series(dtype=object)

    flagged_df = (pd.concat(flagged_frames, ignore_index=True)
                  if flagged_frames else raw_df.iloc[0:0].assign(validation_issue=[]))

    report = {
        'total_raw': total_raw,
        'null_required': int(len(null_rows)),
        'invalid_amount': int(invalid_amount_mask.sum()),
        'negative_amount': negative_count,
        'invalid_date': int(invalid_date_mask.sum()),
        'date_reformatted': date_reformatted,
        'duplicate_id': int(dup_mask.sum()),
        'total_clean': len(df),
    }
    return df, report, flagged_df


In [ ]:
# ===== 4) FRONTEND + DASHBOARD =====
OUT_PATH = '/content/cleaned_transactions.csv'
df = None            # ข้อมูลที่ผ่านการทำความสะอาดแล้ว
df_flagged = None    # รายการที่ถูกตัด/แก้ระหว่าง validation


def fig_top_count(d):
    d = d[d['flow'] == 'expense']
    fig, ax = plt.subplots(figsize=(9, 5))
    top = d['description_standard'].value_counts().head(10)
    sns.barplot(x=top.values, y=top.index, hue=top.index, palette='viridis', legend=False, ax=ax)
    ax.set_title('10 อันดับร้านค้าที่จ่ายเงินบ่อยที่สุด (ไม่รวมรายรับ)')
    ax.set_xlabel('จำนวนครั้ง'); ax.set_ylabel('')
    fig.tight_layout()
    return fig


def fig_top_amount(d):
    d = d[d['flow'] == 'expense']
    fig, ax = plt.subplots(figsize=(9, 5))
    top = d.groupby('description_standard')['amount'].sum().sort_values(ascending=False).head(10)
    sns.barplot(x=top.values, y=top.index, hue=top.index, palette='rocket', legend=False, ax=ax)
    ax.set_title('10 อันดับร้านค้าที่ใช้เงินมากที่สุด (ไม่รวมรายรับ)')
    ax.set_xlabel('ยอดรวม (บาท)'); ax.set_ylabel('')
    fig.tight_layout()
    return fig


def fig_pie(d):
    expenses = d[d['flow'] == 'expense']
    summary = expenses.groupby('category')['amount'].sum()
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.pie(summary.values, labels=summary.index, autopct='%1.1f%%',
           startangle=140, colors=sns.color_palette('pastel'))
    ax.set_title('สัดส่วนรายจ่ายแยกตามหมวดหมู่')
    fig.tight_layout()
    return fig


def fig_trend(d):
    d = d.dropna(subset=['date_cleaned']).copy()
    d = d[d['flow'].isin(['income', 'expense'])]
    d['year_month'] = d['date_cleaned'].dt.to_period('M').astype(str)
    d['flow_label'] = d['flow'].map({'income': 'รายรับ', 'expense': 'รายจ่าย'})
    trend = d.groupby(['year_month', 'flow_label'])['amount'].sum().reset_index()

    fig, ax = plt.subplots(figsize=(11, 5))
    sns.lineplot(data=trend, x='year_month', y='amount', hue='flow_label',
                 marker='o', linewidth=2.5,
                 palette={'รายรับ': 'green', 'รายจ่าย': 'red'}, ax=ax)
    ax.set_title('แนวโน้มรายรับ-รายจ่ายรายเดือน')
    ax.set_xlabel('เดือน-ปี'); ax.set_ylabel('ยอดรวม (บาท)')
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    fig.tight_layout()
    return fig


def fig_income(d):
    """กราฟรายรับ: ซ้าย = รายรับรายเดือน, ขวา = รายรับแยกตามแหล่งที่มา (เฉพาะหมวด Income)"""
    inc = d[d['flow'] == 'income'].copy()
    if inc.empty:
        return None
    inc['year_month'] = inc['date_cleaned'].dt.to_period('M').astype(str)
    monthly = inc.groupby('year_month')['amount'].sum()
    by_source = (inc.groupby('description_standard')['amount'].sum()
                 .sort_values(ascending=False).head(10))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.barplot(x=monthly.index, y=monthly.values, color='seagreen', ax=axes[0])
    axes[0].set_title('รายรับรายเดือน')
    axes[0].set_xlabel('เดือน-ปี'); axes[0].set_ylabel('ยอดรวม (บาท)')
    plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')

    sns.barplot(x=by_source.values, y=by_source.index, hue=by_source.index,
                palette='Greens_r', legend=False, ax=axes[1])
    axes[1].set_title('รายรับแยกตามแหล่งที่มา')
    axes[1].set_xlabel('ยอดรวม (บาท)'); axes[1].set_ylabel('')
    fig.tight_layout()
    return fig


def fig_validation_summary(report):
    labels = {
        'null_required': 'ค่าว่าง',
        'invalid_amount': 'จำนวนเงินผิดรูปแบบ',
        'negative_amount': 'จำนวนเงินติดลบ',
        'date_reformatted': 'วันที่ที่ถูกแปลง format',
        'duplicate_id': 'transaction_id ซ้ำ',
    }
    vals = {labels[k]: report[k] for k in labels}
    fig, ax = plt.subplots(figsize=(8, 4.5))
    # สีแดง = มีปัญหา/ถูกตัด, สีส้ม = แก้ให้อัตโนมัติ (แปลง format วันที่), สีเขียว = ไม่พบ
    colors = []
    for k, v in vals.items():
        if v == 0:
            colors.append('#2ca02c')
        elif k == 'วันที่ที่ถูกแปลง format':
            colors.append('#ff7f0e')
        else:
            colors.append('#d62728')
    sns.barplot(x=list(vals.values()), y=list(vals.keys()), hue=list(vals.keys()),
                palette=colors, legend=False, ax=ax)
    ax.set_title('จำนวนรายการที่พบปัญหา/ถูกแก้ไข แยกตามประเภท')
    ax.set_xlabel('จำนวนแถว')
    fig.tight_layout()
    return fig


def build_validation_report_md(report):
    pct_clean = (report['total_clean'] / report['total_raw'] * 100) if report['total_raw'] else 0
    return (f"### ผลการตรวจสอบคุณภาพข้อมูล (Data Validation)\n"
            f"- จำนวนแถวดิบทั้งหมด: **{report['total_raw']:,}**\n"
            f"- ผ่านการตรวจสอบและใช้งานได้: **{report['total_clean']:,}** แถว "
            f"({pct_clean:.1f}%)\n"
            f"- ค่าว่างในคอลัมน์จำเป็น: **{report['null_required']:,}** แถว (ตัดออก)\n"
            f"- จำนวนเงินไม่ใช่ตัวเลข: **{report['invalid_amount']:,}** แถว (ตัดออก)\n"
            f"- จำนวนเงินติดลบ: **{report['negative_amount']:,}** แถว "
            f"{NEGATIVE_NOTE[NEGATIVE_POLICY]}\n"
            f"- แปลง format วันที่เป็น YYYY-MM-DD: **{report['date_reformatted']:,}** แถว\n"
            + (f"- วันที่ผิดรูปแบบจนแปลงไม่ได้: **{report['invalid_date']:,}** แถว (ตัดออก)\n"
               if report['invalid_date'] else "") +

            f"- transaction_id ซ้ำ: **{report['duplicate_id']:,}** แถว (ตัดออก เก็บรายการแรกไว้)")


def process_csv(file):
    """รับไฟล์จากหน้าเว็บ → ตรวจสอบคุณภาพข้อมูล → รัน pipeline → แสดง Dashboard"""
    global df, df_flagged
    blank = (None,) * 11

    if file is None:
        return ('กรุณาเลือกไฟล์ CSV ก่อนกด Submit', '') + blank

    path = file if isinstance(file, str) else file.name
    try:
        raw = read_csv_safe(path)
    except Exception as e:
        return (f'อ่านไฟล์ไม่สำเร็จ: {e}', '') + blank

    missing = [c for c in REQUIRED_COLS if c not in raw.columns]
    if missing:
        return (f'ไฟล์ขาดคอลัมน์: {", ".join(missing)}\n'
                f'คอลัมน์ที่พบในไฟล์: {", ".join(map(str, raw.columns))}', '') + blank

    try:
        d, report, flagged = validate_and_clean(raw)
        df, df_flagged = d, flagged

        if len(d) == 0:
            return ('ไม่มีข้อมูลผ่านการตรวจสอบเลย กรุณาตรวจสอบไฟล์ต้นฉบับ '
                     '(ดูรายละเอียดที่แท็บ Validation)', '',
                    None, None, None, None, None,
                    build_validation_report_md(report), fig_validation_summary(report),
                    flagged if len(flagged) else None, None, None, None)

        income = d.loc[d['flow'] == 'income', 'amount'].sum()
        expense = d.loc[d['flow'] == 'expense', 'amount'].sum()
        balance = income - expense                        # รายรับ (รวมเงินโอนเข้า) - รายจ่าย (รวมเงินโอนออก)
        top_cat = (d[d['flow'] == 'expense']
                   .groupby('category')['amount'].sum().sort_values(ascending=False))
        top_line = (f"- หมวดที่ใช้เงินมากที่สุด: **{top_cat.index[0]}** ({top_cat.iloc[0]:,.2f} บาท)"
                    if len(top_cat) else "- หมวดที่ใช้เงินมากที่สุด: ไม่มีรายจ่ายในข้อมูล")

        kpi = (f"### สรุปภาพรวม\n"
               f"- รายรับรวม (รวมเงินโอนเข้า): **{income:,.2f}** บาท\n"
               f"- รายจ่ายรวม (รวมเงินโอนออก): **{expense:,.2f}** บาท\n"
               f"- รายรับ - รายจ่าย (เงินสดเปลี่ยนแปลงสุทธิ): **{balance:,.2f}** บาท\n"
               f"- จำนวนรายการที่ใช้งานได้: **{len(d):,}** รายการ "
               f"(จากทั้งหมด {report['total_raw']:,} แถว)\n"
               f"{top_line}")

        cat_table = top_cat.reset_index().rename(
            columns={'category': 'หมวดหมู่', 'amount': 'ยอดรวม (บาท)'})
        cat_table['ยอดรวม (บาท)'] = cat_table['ยอดรวม (บาท)'].round(2)

        final_cols = ['transaction_id', 'date_cleaned', 'description_standard', 'category', 'amount']
        clean_table = d[final_cols].sort_values('date_cleaned')
        clean_table.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

        validation_md = build_validation_report_md(report)
        val_fig = fig_validation_summary(report)
        flagged_display = flagged if len(flagged) else None

        status = f'ประมวลผลสำเร็จ: {len(d):,} รายการผ่านการตรวจสอบ พร้อมแสดง Dashboard ด้านล่าง'
        return (status, kpi, cat_table,
                fig_trend(d), fig_pie(d), fig_top_amount(d), fig_top_count(d),
                validation_md, val_fig, flagged_display,
                clean_table.head(100), OUT_PATH, fig_income(d))
    except Exception as e:
        return (f'ประมวลผลไม่สำเร็จ: {type(e).__name__}: {e}', '') + blank


with gr.Blocks(title='Dashboard รายรับรายจ่าย', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# ระบบกำกับรายรับ-รายจ่าย')
    gr.Markdown('อัปโหลดไฟล์ CSV แล้วกด Submit เพื่อดู Dashboard')

    with gr.Row():
        file_in = gr.File(label='ไฟล์ CSV', file_types=['.csv'], scale=3)
        submit = gr.Button('Submit', variant='primary', scale=1)

    status = gr.Textbox(label='สถานะ', lines=2)

    with gr.Tabs():
        with gr.Tab('ภาพรวม'):
            kpi_md = gr.Markdown()
            cat_df = gr.Dataframe(label='ยอดรวมแยกตามหมวดหมู่')
        with gr.Tab('แนวโน้มรายเดือน'):
            p_trend = gr.Plot()
        with gr.Tab('รายรับ'):
            p_income = gr.Plot()
        with gr.Tab('สัดส่วนรายจ่าย'):
            p_pie = gr.Plot()
        with gr.Tab('ร้านค้า'):
            p_amount = gr.Plot()
            p_count = gr.Plot()
        with gr.Tab('ตรวจสอบคุณภาพข้อมูล (Validation)'):
            val_md = gr.Markdown()
            p_validation = gr.Plot()
            flagged_df_out = gr.Dataframe(label='รายการที่ถูกตัด/แก้ไข พร้อมเหตุผล (validation_issue)')
        with gr.Tab('ข้อมูลที่ทำความสะอาดแล้ว'):
            clean_df = gr.Dataframe(label='100 รายการแรก')
            download = gr.File(label='ดาวน์โหลดไฟล์ CSV ที่ Clean แล้ว')

    submit.click(
        fn=process_csv,
        inputs=file_in,
        outputs=[status, kpi_md, cat_df, p_trend, p_pie, p_amount, p_count,
                 val_md, p_validation, flagged_df_out, clean_df, download, p_income],
    )

demo.launch(share=True, debug=False)


/tmp/ipykernel_2404/4230189524.py:194: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='Dashboard รายรับรายจ่าย', theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://632ba8b3aae7dacde5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
